#### Import Packages

In [ ]:
import numpy as np
np.random.seed(0)   # choose any integer
import os
import time, joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils import shuffle
from sklearn.metrics import precision_score, recall_score, log_loss

import tensorflow as tf
tf.random.set_seed(0)
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Softmax, Multiply, Lambda, Flatten, MultiHeadAttention, LayerNormalization, Conv1D, GlobalAveragePooling1D, ReLU, add
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import load_model

from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
print(tf.__version__)

#### Read Data

In [ ]:
def read_data(folder_path):
    dataframes = {} # Create an empty dictionary to store DataFrames
    csv_files = sorted([f for f in os.listdir(folder_path) if f.endswith('.csv')]) # List all CSV files in the folder
    
    for csv_file in csv_files:
        attack = os.path.splitext(csv_file)[0] 
        file_path = os.path.join(folder_path, csv_file)
        dataframes[attack] = pd.read_csv(file_path)
    return dataframes

In [ ]:
    folder_path = '../Datasets/sequences_mc'
    csv_files = sorted([f for f in os.listdir(folder_path) if f.endswith('.csv')])
    print(csv_files)

#### Auxiliary Methods

In [ ]:
def scale_df(data,features):
    #scaler = StandardScaler()
    #scaler = MinMaxScaler(feature_range=(0, 1))
    scaler = RobustScaler()
    data[features] = scaler.fit_transform(data[features]) # normalize the above features
    return data, scaler

In [ ]:
def lstm_model(input_shape, lstm_units=32, dropout_rate=0.2, learning_rate=0.01):
    inputs = Input(shape=input_shape, name="input_layer")  # Input Layer (batch_size, time_steps, features)
    lstm_out = LSTM(lstm_units, return_sequences=False, name="lstm_layer")(inputs)   # LSTM Layer  (batch_size, lstm_units)
    x = Dropout(dropout_rate, name="dropout_after_lstm")(lstm_out)
    outputs = Dense(1, activation="sigmoid", name="output_layer")(x)  # # Classification Layer (binary)

    # Compile Model
    model = Model(inputs=inputs, outputs=outputs)
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    loss_fn = tf.keras.losses.BinaryCrossentropy(from_logits=False)
    model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])
    #model.compile(optimizer="adam",loss=weighted_bce, metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

    return model

In [ ]:
def lstm_model(input_shape, lstm_units=32, dropout_rate=0.2, learning_rate=0.01):
    inputs = Input(shape=input_shape, name="input_layer")

    # LSTM with recurrent dropout only
    lstm_out = LSTM(
        lstm_units,
        return_sequences=False,
        dropout=0.0,              # no input dropout
        recurrent_dropout=0.2,    # dropout in recurrent connections
        name="lstm_layer"
    )(inputs)

    # Dropout after the LSTM output
    x = Dropout(dropout_rate, name="dropout_after_lstm")(lstm_out)

    # Final classifier
    outputs = Dense(1, activation="sigmoid", name="output_layer")(x)

    # Compile
    model = Model(inputs=inputs, outputs=outputs)
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    loss_fn = tf.keras.losses.BinaryCrossentropy(from_logits=False)

    model.compile(
        optimizer=optimizer,
        loss=loss_fn,
        metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
    )
    return model


In [ ]:
# Create an encoder-only transformer based model (got from keras website: https://keras.io/examples/timeseries/timeseries_classification_transformer/)
def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    # Multi-Head Self Attention + Norm
    x = MultiHeadAttention(
        key_dim=head_size, num_heads=num_heads, dropout=dropout
    )(inputs, inputs)
    x = Dropout(dropout)(x)
    x = LayerNormalization(epsilon=1e-6)(x)
    res = x + inputs

    # Feed Forward
    x = Conv1D(filters=ff_dim, kernel_size=1, activation="relu")(res)
    x = Dropout(dropout)(x)
    x = Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    x = LayerNormalization(epsilon=1e-6)(x)
    return x + res

def build_transformer_classifier(input_shape, head_size=64, num_heads=2, ff_dim=64,
                                 num_blocks=2, dropout=0.1):
    inputs = Input(shape=input_shape)  # (seq_len, features)
    x = inputs

    # Stack Transformer blocks
    for _ in range(num_blocks):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)

    # Global Average Pooling
    x = GlobalAveragePooling1D()(x)
    x = Dropout(dropout)(x)
    outputs = Dense(1, activation="sigmoid")(x)


    # Compile Model
    model = Model(inputs=inputs, outputs=outputs)
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
    #model.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])
    model.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])
    return model


# param_grids = {
#     1: {  # Transformer (Avg)
#         "head_size": [32, 64],
#         "num_heads": [2, 4],
#         "ff_dim": [64, 128],
#         "dropout": [0.1, 0.2]

In [ ]:
import math
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv1D, ReLU, Dropout, add, GlobalAveragePooling1D, Dense
from tensorflow.keras.models import Model

def residual_block(x, filters, kernel_size, dropout, dilation_rate=1):
    prev_x = x

    conv1 = Conv1D(filters, kernel_size,
                   padding="causal", dilation_rate=dilation_rate)(x)
    conv1 = ReLU()(conv1)
    conv1 = Dropout(dropout)(conv1)

    conv2 = Conv1D(filters, kernel_size,
                   padding="causal", dilation_rate=dilation_rate)(conv1)
    conv2 = ReLU()(conv2)
    conv2 = Dropout(dropout)(conv2)

    if prev_x.shape[-1] != filters:
        prev_x = Conv1D(filters, 1, padding="same")(prev_x)

    out = add([conv2, prev_x])
    out = ReLU()(out)
    return out

def build_tcn(num_features, max_seq_len=10, filters=32, kernel_size=2, dropout=0.1):
    # compute minimum num_blocks to cover max_seq_len
    num_blocks = math.ceil(math.log2(max_seq_len))
    RF = 1 + (kernel_size - 1) * (2**num_blocks - 1)
    if RF < max_seq_len:
        raise ValueError(f"Receptive field ({RF}) < max sequence length ({max_seq_len})")

    inputs = Input(shape=(None, num_features))  # variable length
    x = inputs

    for b in range(num_blocks):
        dilation_rate = 2 ** b
        x = residual_block(x, filters, kernel_size, dropout, dilation_rate=dilation_rate)

    x = GlobalAveragePooling1D()(x)
    outputs = Dense(1, activation="sigmoid")(x)

    model = Model(inputs, outputs)
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
    )

    # ---- print parameters here ----
    print(
        f"Built TCN | RF={RF} | seq_len∈[2,{max_seq_len}] "
        f"| num_features={num_features} | filters={filters} "
        f"| kernel_size={kernel_size} | dropout={dropout}"
    )

    return model



# tcn_grid = {
#     "filters": [32, 64],
#     "kernel_size": [2, 3],
#     "dropout": [0.1, 0.2],
#     "num_blocks": [1, 2, 3]
# }

In [ ]:
def _eval_sklearn(model, X, y):
    """Return loss, acc, prec, rec, report, cm; also proba & yhat if needed"""
    Xf = _flatten3d(X)
    proba = model.predict_proba(Xf)[:, 1]
    yhat  = (proba >= 0.5).astype(int)
    try:
        loss = log_loss(y, proba, eps=1e-15)
    except ValueError:
        loss = float("nan")
    acc  = accuracy_score(y, yhat)
    prec = precision_score(y, yhat, zero_division=0)
    rec  = recall_score(y, yhat, zero_division=0)
    report = classification_report(y, yhat, target_names=["Normal", "Attack"], output_dict=True)
    cm = confusion_matrix(y, yhat)
    return loss, acc, prec, rec, proba, yhat, report, cm

In [ ]:
def _flatten3d(X):
    """(N, T, F) -> (N, T*F)"""
    return X.reshape(len(X), -1)

def _make_model(model_choice, y_train):
    """4 = XGBoost (binary), 5 = KNN (binary)"""
    if model_choice == 4:
        pos = max(1, int((y_train == 1).sum()))
        neg = max(1, int((y_train == 0).sum()))
        spw = neg / pos
        return XGBClassifier(
            n_estimators=500, max_depth=6, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
            objective="binary:logistic", eval_metric="logloss",
            scale_pos_weight=spw, n_jobs=-1, random_state=42
        )
    elif model_choice == 5:
        return KNeighborsClassifier(n_neighbors=10, weights="distance")
    else:
        raise ValueError("model_choice must be 4 (XGBoost) or 5 (KNN) here.")

def _eval_sklearn(model, X, y):
    """Return loss, acc, prec, rec, report, cm; also proba & yhat if needed"""
    Xf = _flatten3d(X)
    proba = model.predict_proba(Xf)[:, 1]
    yhat  = (proba >= 0.5).astype(int)
    try:
        loss = log_loss(y, proba)
    except ValueError:
        loss = float("nan")
    acc  = accuracy_score(y, yhat)
    prec = precision_score(y, yhat, zero_division=0)
    rec  = recall_score(y, yhat, zero_division=0)
    report = classification_report(y, yhat, target_names=["Normal", "Attack"], output_dict=True)
    cm = confusion_matrix(y, yhat)
    return loss, acc, prec, rec, proba, yhat, report, cm


#### Training

In [ ]:
import gc, tensorflow as tf

def cleanup():
    gc.collect()
    tf.keras.backend.clear_session()

In [ ]:
### Resample
# pip install imbalanced-learn
from imblearn.over_sampling import SMOTE
import numpy as np
from imblearn.under_sampling import TomekLinks
from imblearn.pipeline import Pipeline

def smote_windows_class1(X_train, y_train, target_pos_rate=0.20, random_state=42):
    """
    X_train: (N, 3, 14)
    y_train: (N,) in {0,1}
    target_pos_rate: desired fraction of positives after resampling (e.g., 0.20 = 20%)
    returns X_res, y_res with only class-1 oversampled.
    """
    y_train = y_train.astype(int)
    pos = int((y_train == 1).sum())
    neg = int((y_train == 0).sum())
    if pos < 2:                    # SMOTE needs at least k_neighbors+1 positives
        return X_train, y_train

    # choose k_neighbors safely
    k = min(5, pos - 1)

    # target minority/majority ratio after resampling:
    # r = P' / (P' + N)  ->  P'/N = r/(1-r)
    r = float(target_pos_rate)
    ratio = r / (1.0 - r)

    # OR specify exact desired positive count:
    desired_pos = int(ratio * neg)         # P' = ratio * N

    # Flatten (N, 3*14)
    X_flat = X_train.reshape(X_train.shape[0], -1)

    # Oversample ONLY class-1 to desired count
    smote = SMOTE(
        sampling_strategy={1: desired_pos},  # only class 1 is synthesized
        k_neighbors=k,
        random_state=random_state
    )
    tomek = TomekLinks(sampling_strategy='auto')
    pipe = Pipeline([("smote", smote), ("tomek", tomek)])
    X_res_flat, y_res = pipe.fit_resample(X_flat, y_train)
    X_res = X_res_flat.reshape(-1, X_train.shape[1], X_train.shape[2])
    return X_res, y_res


In [ ]:
import numpy as np
from sklearn.metrics import roc_curve, confusion_matrix

def training(sequences, features, ws, previous_data_ratio, csv_path, model_choice):
    class_weights = {0: 1.0, 1:1.0}
    save_dir = "saved_models"
    os.makedirs(save_dir, exist_ok=True)  # Create folder if not exists
    attacks = ['SYN', 'ICMP', 'UDP', 'DNS', 'GTPU'] # Order of attacks: SYN - ICMP - UDP - DNS - GTPU 
    #attacks = ['GTPU']
    models = [] # store models for each day
    results = [] # store results of cumulative evaluations (up to day x)
    per_day_results = []  # Store per-day evaluations
    future_days_evaluation = {}  # Store future days' evaluations
    past_evaluation_results = {}  # Store evaluations on previous days' test sets
    cumulative_X_test, cumulative_y_test, cumulative_y_multi_test = np.array([]), np.array([]), np.array([]) # Initialize cumulative test sets
    test_sets = {} # Initialize a dictionary to store test sets for future and past evaluations
    previous_normal_data, previous_attack_data = [], [] # Initialize buffers for sampled data 
    is_sklearn = model_choice in (4, 5)
    total_data_prep = 0.0
    total_training = 0.0
    total_time = 0.0
    
    # Training and fine-tuning loop
    for i, attack in enumerate(attacks):
        total_start = time.time()
        data_start = time.time()

        X_train, X_test, y_train, y_test, y_multi_train, y_multi_test = sequences[attack]
        # Training set
        #unique, counts = np.unique(y_train, return_counts=True)
        #print("Train label counts:", dict(zip(unique, counts)))
        
        # Test set
        unique, counts = np.unique(y_test, return_counts=True)
        print("Test label counts:", dict(zip(unique, counts)))
        print(X_train.shape)
        X_train, y_train = smote_windows_class1(X_train, y_train, target_pos_rate=0.10 , random_state=42) # 
        unique, counts = np.unique(y_train, return_counts=True)
        print("Train label counts:", dict(zip(unique, counts)))
        test_sets[i] = (X_test, y_test, y_multi_test) # Save current test set for evaluations
        # Concatenate cumulative test sets for evaluation
        cumulative_X_test = np.concatenate([cumulative_X_test, X_test]) if cumulative_X_test.size else X_test
        cumulative_y_test = np.concatenate([cumulative_y_test, y_test]) if cumulative_y_test.size else y_test
        cumulative_y_multi_test = np.concatenate([cumulative_y_multi_test, y_multi_test]) if cumulative_y_multi_test.size else y_multi_test
        # Stratified mixing of previous data
        if i > 0 and (previous_normal_data or previous_attack_data):
            print(f"Mixing {previous_data_ratio * 100:.0f}% of previous days' data with current day's data")

            # Use only the last stored array instead of concatenating all
            sampled_X_normal = previous_normal_data[-1] 
            sampled_X_attack = previous_attack_data[-1] 
    
            # Combine normal and attack samples
            sampled_previous_X = np.concatenate((sampled_X_normal, sampled_X_attack), axis=0)
            sampled_previous_y = np.concatenate((np.zeros(len(sampled_X_normal)), np.ones(len(sampled_X_attack))), axis=0)
    
            # Combine sampled previous data with current day's training data
            mixed_X_train = np.concatenate((X_train, sampled_previous_X), axis=0)
            mixed_y_train = np.concatenate((y_train, sampled_previous_y), axis=0)
            mixed_X_train, mixed_y_train = shuffle(mixed_X_train, mixed_y_train, random_state=42) # Shuffle the combined data
        else:
            mixed_X_train = X_train
            mixed_y_train = y_train
        data_end = time.time()
        data_prep_time = data_end - data_start
        
        # Train or fine-tune the model
        # early stopping
        #early_stopping = EarlyStopping( monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)
        early_stopping = EarlyStopping(monitor='val_recall', mode='max',patience=5, restore_best_weights=True)
        train_start = time.time()
        if i == 0:  # Initial training
            if model_choice == 0:
                print("Training LSTM model...")
                model = lstm_model(X_train.shape[1:])
            elif model_choice == 1:
                print("Training Transformer (GlobalAvg) model...")
                model = build_transformer_classifier(X_train.shape[1:])
            elif model_choice == 3:
                print("Training TCN model...")
                seq_len, num_features = X_train.shape[1:]
                model = build_tcn(num_features, max_seq_len=seq_len, filters=32, kernel_size=2, dropout=0.1)
            elif model_choice in (4, 5):
                print("Training XGBoost model..." if model_choice == 4 else "Training KNN model...")
                model = _make_model(model_choice, mixed_y_train)
            else:
                raise ValueError("Invalid model_choice! Use 0=LSTM, 1=Transformer, 3=TCN, 4=XGB, 5=KNN.")
        
            if is_sklearn:
                model.fit(_flatten3d(mixed_X_train), mixed_y_train)
            else:
                print(model.summary())
                model.fit(
                    mixed_X_train, mixed_y_train,
                    epochs=100, batch_size=64, validation_split=0.2,
                    class_weight=class_weights, callbacks=[early_stopping]
                )
        else:  # Fine-tune / Refit
            if is_sklearn:
                # Refit on new mixed data; recreate XGB each day to refresh scale_pos_weight
                if model_choice == 4:
                    model = _make_model(model_choice, mixed_y_train)
                model.fit(_flatten3d(mixed_X_train), mixed_y_train)
            else:
                model.fit(
                    mixed_X_train, mixed_y_train,
                    epochs=100, batch_size=64, validation_split=0.2,
                    class_weight=class_weights, callbacks=[early_stopping]
                )
        
        train_end = time.time()
        training_time = train_end - train_start


        total_end = time.time()
        day_total = total_end - total_start
        
        # Accumulate
        total_data_prep += data_prep_time
        total_training += training_time
        total_time += day_total
        
        print(f"Day {i+1} ({attack}) timings:")
        print(f"  Data prep time: {data_prep_time:.2f} seconds")
        print(f"  Training time: {training_time:.2f} seconds")
        print(f"  Total time: {total_time:.2f} seconds\n")
    
        # Save model after each day’s training
        model_path = os.path.join(save_dir, f"day_{i+1}_{attack}.h5")
        # Save model after each day’s training
        if is_sklearn:
            model_path = os.path.join(save_dir, f"day_{i+1}_{attack}.pkl")
            joblib.dump(model, model_path)
        else:
            model_path = os.path.join(save_dir, f"day_{i+1}_{attack}.h5")
            model.save(model_path)
        print(f"Model for Day {i+1} ({attack}) saved at {model_path}")
        
        # Sampling data from the current day's training set for future use
        print(f"Sampling data from current training set for future use...")    
        shuffled_normal_X = shuffle(mixed_X_train[mixed_y_train == 0], random_state=42) # Shuffle and sample normal data
        num_normal_samples = int(previous_data_ratio * len(shuffled_normal_X)) # get ratio of normal data
        previous_normal_data.append(shuffled_normal_X[:num_normal_samples])

        shuffled_attack_X = shuffle(mixed_X_train[mixed_y_train == 1], random_state=42) # Shuffle and sample attack data
        num_attack_samples = int(previous_data_ratio * len(shuffled_attack_X)) # get ratio of attack data
        previous_attack_data.append(shuffled_attack_X[:num_attack_samples])

        results_list = []
        print(f"Evaluating Model of Day {i + 1} on Day {i + 1} Test Set...")
        if is_sklearn:
            per_loss, per_acc, per_prec, per_rec, _, _, class_report, conf_matrix = _eval_sklearn(model, X_test, y_test)
        else:
            per_loss, per_acc, per_prec, per_rec = model.evaluate(X_test, y_test, verbose=0)
            predictions = model.predict(X_test, verbose=0).flatten()
            predicted_labels = (predictions >= 0.5).astype(int)
            class_report = classification_report(y_test, predicted_labels, target_names=["Normal", "Attack"], output_dict=True)
            conf_matrix = confusion_matrix(y_test, predicted_labels)
        results_list.append([i + 1, "Per-Day", ws, per_loss, per_acc, per_prec, per_rec, class_report, conf_matrix.tolist()])

        if is_sklearn:
            cumulative_loss, cumulative_accuracy, cumulative_precision, cumulative_recall, _, _, class_report, conf_matrix = _eval_sklearn(model, cumulative_X_test, cumulative_y_test)
        else:
            cumulative_loss, cumulative_accuracy, cumulative_precision, cumulative_recall = model.evaluate(cumulative_X_test, cumulative_y_test, verbose=0)
            predictions = model.predict(cumulative_X_test, verbose=0).flatten()
            predicted_labels = (predictions >= 0.5).astype(int)
            class_report = classification_report(cumulative_y_test, predicted_labels, target_names=["Normal", "Attack"], output_dict=True)
            conf_matrix = confusion_matrix(cumulative_y_test, predicted_labels)
        results_list.append([i + 1, "Cumulative", ws, cumulative_loss, cumulative_accuracy, cumulative_precision, cumulative_recall, class_report, conf_matrix.tolist()])

        if i > 0:
            for prev_day in range(i):
                prev_X_test, prev_y_test, _ = test_sets[prev_day]
                if is_sklearn:
                    prev_loss, prev_acc, prev_prec, prev_rec, _, _, class_report, conf_matrix = _eval_sklearn(model, prev_X_test, prev_y_test)
                else:
                    prev_loss, prev_acc, prev_prec, prev_rec = model.evaluate(prev_X_test, prev_y_test, verbose=0)
                    predictions = model.predict(prev_X_test, verbose=0).flatten()
                    predicted_labels = (predictions >= 0.5).astype(int)
                    class_report = classification_report(prev_y_test, predicted_labels, target_names=["Normal", "Attack"], output_dict=True)
                    conf_matrix = confusion_matrix(prev_y_test, predicted_labels)
                results_list.append([i + 1, f"Previous Day {prev_day + 1}", ws, prev_loss, prev_acc, prev_prec, prev_rec, class_report, conf_matrix.tolist()])

        # Append results to CSV file
        df_results = pd.DataFrame(results_list, columns=["Day", "Evaluation Type", "Window Size", "Loss", "Accuracy", "Precision", "Recall", "Classification Report", "Confusion Matrix"])
        df_results.to_csv(csv_path, mode='a', header=not pd.io.common.file_exists(csv_path), index=False)

    # save the cumulative x_test, y_test, and y_test_multiclass
    N, T, F = cumulative_X_test.shape
    flat_X = cumulative_X_test.reshape(N, -1)
    cols_X = [f"{feat}_t{t}" for t in range(T) for feat in features]
    print("Save csv")
    pd.DataFrame(flat_X, columns=cols_X).to_csv("features.csv", index=False)
    pd.DataFrame({"label": cumulative_y_test.astype(int)}).to_csv("labels.csv", index=False)
    pd.DataFrame({"multiclass_label": cumulative_y_multi_test}).to_csv("multiclass_labels.csv", index=False)

    
    print("\nPerforming Future Test Set Evaluations:")
    for i, attack in enumerate(attacks):
        if is_sklearn:
            path = f"saved_models/day_{i+1}_{attack}.pkl"
            m = joblib.load(path)
            def eval_fn(X, y):
                return _eval_sklearn(m, X, y)
        else:
            path = f"saved_models/day_{i+1}_{attack}.h5"
            m = load_model(path, compile=True)
            def eval_fn(X, y):
                loss, acc, prec, rec = m.evaluate(X, y, verbose=0)
                preds = m.predict(X, verbose=0).flatten()
                yhat = (preds >= 0.5).astype(int)
                rep = classification_report(y, yhat, target_names=["Normal", "Attack"], output_dict=True)
                cm = confusion_matrix(y, yhat)
                return loss, acc, prec, rec, preds, yhat, rep, cm
    
        for j in range(i + 1, len(attacks)):
            future_X_test, future_y_test, _ = test_sets[j]
            f_loss, f_acc, f_prec, f_rec, _, _, f_rep, f_cm = eval_fn(future_X_test, future_y_test)
            df_results = pd.DataFrame([[i + 1, f"Future Day {j+1}", ws, f_loss, f_acc, f_prec, f_rec, f_rep, f_cm.tolist()]],
                                      columns=["Day", "Evaluation Type", "Window Size", "Loss", "Accuracy", "Precision", "Recall", "Classification Report", "Confusion Matrix"])
            df_results.to_csv(csv_path, mode='a', header=False, index=False)

            
 

    # ---- FP/FN breakdown by multiclass label for the last trained model ----
    if is_sklearn:
        probs_cum = model.predict_proba(_flatten3d(cumulative_X_test))[:, 1]
    else:
        probs_cum = model.predict(cumulative_X_test, verbose=0).flatten()
    y_pred_cum = (probs_cum >= 0.5).astype(int)

    
    cum_eval = pd.DataFrame({
        "y_true": cumulative_y_test,
        "y_pred": y_pred_cum,
        "multiclass_label": cumulative_y_multi_test
    })
    cum_eval["is_fp"] = (cum_eval["y_true"] == 0) & (cum_eval["y_pred"] == 1)
    cum_eval["is_fn"] = (cum_eval["y_true"] == 1) & (cum_eval["y_pred"] == 0)
    
    print("\n=== False Positives by attack type (last model, cumulative test sets) ===")
    print(cum_eval.loc[cum_eval["is_fp"], "multiclass_label"].value_counts())
    
    print("\n=== False Negatives by attack type (last model, cumulative test sets) ===")
    print(cum_eval.loc[cum_eval["is_fn"], "multiclass_label"].value_counts())
    
    # Optional: per-attack summary table
    per_multi = (
        cum_eval.groupby("multiclass_label")
                .agg(TP=("y_true", lambda s: int(((cum_eval.y_true==1)&(cum_eval.y_pred==1)&(cum_eval.multiclass_label==s.name)).sum())),
                     TN=("y_true", lambda s: int(((cum_eval.y_true==0)&(cum_eval.y_pred==0)&(cum_eval.multiclass_label==s.name)).sum())),
                     FP=("is_fp","sum"),
                     FN=("is_fn","sum"),
                     support_pos=("y_true","sum"),
                     support_all=("y_true","count"))
                .reset_index()
    )
    per_multi["support_neg"] = per_multi["support_all"] - per_multi["support_pos"]
    per_multi["FN_rate_%"] = 100 * per_multi["FN"] / per_multi["support_pos"].replace(0, np.nan)
    per_multi["FP_rate_%"] = 100 * per_multi["FP"] / per_multi["support_neg"].replace(0, np.nan)
    per_multi = per_multi.sort_values(["FN_rate_%","FP_rate_%"], ascending=[False, False])
    
    print("\nPer-attack misclassification summary (cumulative):")
    print(per_multi[["multiclass_label","support_pos","support_neg","TP","TN","FP","FN","FN_rate_%","FP_rate_%"]]
          .round(3).to_string(index=False))

    # Final summary after all attacks
    print("==== Summary across all 5 days ====")
    print(f"Total data prep time: {total_data_prep:.2f} seconds")
    print(f"Total training time: {total_training:.2f} seconds")
    print(f"Overall total time: {total_time:.2f} seconds")

    start_inf = time.perf_counter()
    if is_sklearn:
        _ = model.predict_proba(_flatten3d(X_test))[:, 1]
    else:
        _ = model.predict(X_test, verbose=0)
    end_inf = time.perf_counter()

    
    inference_time = end_inf - start_inf
    print(f"Inference time for batch of size {len(X_test)}: {inference_time*1000:.3f} ms")
    
    avg_instance_time = inference_time / len(X_test)
    print(f"Average per instance: {avg_instance_time*1000:.3f} ms")

#### Main Script

In [ ]:
#folder_path = '../Datasets/Per_Attack_Datasets'
folder_path = '../Datasets/sequences_mc'
dataframes = read_data(folder_path)

features = ['epre','pusch_snr','p_ue','ul_mcs','cqi','ul_bitrate',
            'dl_mcs','dl_retx','ul_tx','dl_tx','ul_retx','dl_bitrate','dl_err','ul_err'] # features in the dataset
num_features = 14 # number of features 
ws =3

In [ ]:
from sklearn.model_selection import train_test_split

# split each dataframe in the dict
splits = {}  # attack -> (X_train, X_test, y_train, y_test)

# save files
output_dir = "sequences_mc_train_test"
os.makedirs(output_dir, exist_ok=True)

test_size = 0.2  # 20% test

for attack, df in dataframes.items():
    # separate features and label
    X = df.drop(columns=["label", "multiclass_label"]).to_numpy(dtype="float32")
    y = df["label"].to_numpy(dtype="int8")
    y_multiclass = df["multiclass_label"].to_numpy()
    X_train, X_test, y_train, y_test, y_multi_train, y_multi_test = train_test_split(
        X, y,y_multiclass , test_size=test_size, stratify=y, random_state=42
    )

    splits[attack] = (X_train, X_test, y_train, y_test, y_multi_train, y_multi_test)
    print(f"{attack}: train {X_train.shape}, test {X_test.shape}")
    print(len(splits[attack]))
    
    train_df = pd.DataFrame(X_train, columns=df.drop(columns=["label", "multiclass_label"]).columns)
    train_df["label"] = y_train
    train_df["multiclass_label"] = y_multi_train

    test_df = pd.DataFrame(X_test, columns=df.drop(columns=["label", "multiclass_label"]).columns)
    test_df["label"] = y_test
    test_df["multiclass_label"] = y_multi_test

    train_df.to_csv(os.path.join(output_dir, f"{attack}_train.csv"), index=False)
    test_df.to_csv(os.path.join(output_dir, f"{attack}_test.csv"), index=False)

    print(f"{attack}: train {train_df.shape}, test {test_df.shape}")

In [ ]:
import numpy as np
# Collect training splits
X_train_all = np.vstack([v[0] for v in splits.values()])   # v[0] = X_train
N = X_train_all.shape[0]
X_train_all_3d = X_train_all.reshape(N,ws,num_features)
X_train_all_rows = X_train_all_3d.reshape(-1,num_features)
#scaler= MinMaxScaler(feature_range=(-1, 1))
scaler= RobustScaler()
#scaler = StandardScaler()
scaler.fit(X_train_all_rows)

In [ ]:
# 2) Transform each attack separately and keep structure
scaled_splits = {}
for attack, (X_tr, X_te, y_tr, y_te, y_mtr, y_mte) in splits.items():
    
    Ntr, Nte = X_tr.shape[0], X_te.shape[0]
    
    # reshape to 3D
    X_tr_3d = X_tr.reshape(Ntr, ws, num_features)
    X_te_3d = X_te.reshape(Nte, ws, num_features)
    
   # collapse time -> (N*ws, num_features), transform, then reshape back
    X_tr_scaled = scaler.transform(X_tr_3d.reshape(-1, num_features)).reshape(Ntr, ws, num_features).astype(np.float32)

    X_te_scaled = scaler.transform(X_te_3d.reshape(-1, num_features)).reshape(Nte, ws, num_features).astype(np.float32)

    # keep per-attack structure
    scaled_splits[attack] = (X_tr_scaled, X_te_scaled, y_tr, y_te, y_mtr, y_mte)

In [ ]:
# (Optional) sanity check
for attack, (X_tr_s, X_te_s, y_tr, y_te, y_mtr, y_mte) in scaled_splits.items():
    print(f"{attack:>8} | train {X_tr_s.shape} | test {X_te_s.shape} | y_tr {y_tr.shape} | y_te {y_te.shape} | y_mtr {y_mtr.shape} | y_mte {y_mte.shape}")

In [ ]:
mcs = [4]  # just one model choice
window_sizes = [3]
ratios = [0.3]
output_dir = "final_experiments"
os.makedirs(output_dir, exist_ok=True)  # Creates the directory if it doesn't exist
for ws in window_sizes:
    #sequences = precompute_sequences(dataframes, features, ws)
    for r in ratios:
        for mc in mcs:
            for iteration in range(1,3): 
                output_path = f"{output_dir}/experiments_{ws}_{r}_{iteration}_model_{mc}_new.csv"
                cleanup()
                training(scaled_splits, features, ws, r, output_path, model_choice=mc)

In [ ]:
mcs = [5]  # just one model choice
window_sizes = [3]
ratios = [0.3]
output_dir = "final_experiments"
os.makedirs(output_dir, exist_ok=True)  # Creates the directory if it doesn't exist
for ws in window_sizes:
    #sequences = precompute_sequences(dataframes, features, ws)
    for r in ratios:
        for mc in mcs:
            for iteration in range(1,3): 
                output_path = f"{output_dir}/experiments_{ws}_{r}_{iteration}_model_{mc}_new.csv"
                cleanup()
                training(scaled_splits, features, ws, r, output_path, model_choice=mc)

In [ ]:
mcs = [4]  # just one model choice
window_sizes = [3]
ratios = [0.3]
output_dir = "final_experiments"
os.makedirs(output_dir, exist_ok=True)  # Creates the directory if it doesn't exist
for ws in window_sizes:
    #sequences = precompute_sequences(dataframes, features, ws)
    for r in ratios:
        for mc in mcs:
            for iteration in range(1,3): 
                output_path = f"{output_dir}/experiments_{ws}_{r}_{iteration}_model_{mc}_new.csv"
                cleanup()
                training(scaled_splits, features, ws, r, output_path, model_choice=mc)

In [ ]:
mcs = [5]  # just one model choice
window_sizes = [3]
ratios = [0.3]
output_dir = "final_experiments"
os.makedirs(output_dir, exist_ok=True)  # Creates the directory if it doesn't exist
for ws in window_sizes:
    #sequences = precompute_sequences(dataframes, features, ws)
    for r in ratios:
        for mc in mcs:
            for iteration in range(1,3): 
                output_path = f"{output_dir}/experiments_{ws}_{r}_{iteration}_model_{mc}_new.csv"
                cleanup()
                training(scaled_splits, features, ws, r, output_path, model_choice=mc)